# Simulating and visualizing a system of particles with interactions between pairs and triplets

## Documentation for Particle Simulation Code

This Python code simulates the movement of particles and their interactions within a 2D space using Pygame for visualization. It incorporates both **pairwise** and **triplet interactions** among particles and handles different boundary conditions (either **hard boundaries** or **periodic boundaries**). Additionally, the code records particle positions at specified time intervals and stores the data in a CSV file. Below is an explanation of the main components of the code:

---

### **Simulation Parameters**

- **boundary**: Defines the boundary condition of the simulation. The possible values are:
  - `"Hard"`: Particles reflect off the edges of the screen.
  - `"Periodic"`: Particles wrap around to the opposite side when they exit one side of the screen.
- **num_particles**: The number of particles in the simulation.
- **width, height**: The dimensions of the simulation area.
- **interaction_distance**: The distance at which pairwise interactions occur between particles.
- **triplet_interaction_distance**: The distance at which triplet interactions occur.
- **speed**: The speed at which particles move.
- **dt**: The time step for the simulation.
- **simulation_duration**: The total simulation time in arbitrary time units.
- **time_step_record**: The frequency with which particle positions are recorded (in simulation time).

---

### **Particle Class**

The `Particle` class represents each particle in the simulation and contains the following attributes and methods:

- **Attributes**:
  - `position`: A 2D numpy array holding the particle’s position (x, y).
  - `velocity`: A 2D numpy array representing the particle’s velocity.
  - `move_time`: Random time during which the particle moves before resting.
  - `rest_time`: Random time the particle rests.
  - `timer`: Tracks the remaining time the particle will continue its current state (moving or resting).
  - `moving`: A boolean flag indicating whether the particle is moving or resting.

- **Methods**:
  - `update()`: This method updates the position of the particle based on its velocity and handles both movement and resting states. It also ensures particles stay within the boundaries defined by the simulation (either through hard boundaries or periodic boundaries).

---

### **Interaction Functions**

1. **pairwise_interact(particles)**:
   - This function computes the interaction forces between pairs of particles. If two particles are within the `interaction_distance`, they interact. The force is proportional to the distance between them and inversely affects their velocities.

2. **triplet_interact(particles)**:
   - This function computes the interaction forces among triplets of particles. It checks if all three particles in a triplet are within the `triplet_interaction_distance`. If so, the particles interact by moving away from their centroid, with the interaction strength defined by `triplet_interaction_strength`.

---

### **Simulation Loop**

- **Main loop**: 
   - The simulation runs for `simulation_duration` time steps. In each step:
     - Particle positions are updated based on their velocities.
     - Pairwise and triplet interactions are handled.
     - Particle positions are recorded at intervals specified by `time_step_record`.
     - The screen is updated with the new particle positions and interactions.

- **Recording Positions**: 
   - At each time step, the positions of the particles are recorded in a `pandas` DataFrame. This is then saved to a CSV file, `particle_positions.csv`.

---

### **Output**

The simulation will output a CSV file that contains the positions and velocities of all particles over time. The columns in the CSV file are as follows:

- **time**: The current time step.
- **x1, y1, x2, y2, ..., xn, yn**: The x and y coordinates for each particle at that time step (where `n` is the number of particles).
- **vx1, vy1, vx2, vy2, ..., vxn, vyn**: The x and y velocities for each particle at that time step (where `n` is the number of particles).

## The code

In [1]:
import random
import pygame  # type: ignore
import numpy as np  # type: ignore
import pandas as pd  # type: ignore

# Simulation parameters
boundary = "Hard"  # "Periodic", "Hard"
num_particles = 4
width, height = 240, 135
interaction_distance = 25
triplet_interaction_distance = 100  # Distance for triplet interactions
speed = 15
dt = 0.1

# Total simulation time in units
simulation_duration = 3603

# Time step for recording positions
time_step_record = 3

# Strength for interactions
pairwise_interaction_strength = -0.1  # Repulsive if >0, attractive if <0
triplet_interaction_strength = 0.01  # Strength for triplet interactions

# Resting and moving times
move_time_min = 1
move_time_max = 4
rest_time_min = 5
rest_time_max = 5

# Initialize Pygame
pygame.init()
screen = pygame.display.set_mode((width, height))
clock = pygame.time.Clock()


if boundary == "Hard":
    # Particle class with hard boundary conditions
    class Particle:
        def __init__(self):
            self.position = np.array([random.uniform(0, width), random.uniform(0, height)])
            self.velocity = np.zeros(2)
            self.move_time = random.uniform(move_time_min, move_time_max)
            self.rest_time = random.uniform(rest_time_min, rest_time_max)
            self.timer = self.move_time
            self.moving = True
    
        def update(self):
            if self.moving:
                self.position += self.velocity * dt
                self.timer -= dt
    
                # Reverse velocity and keep position within boundaries when hitting walls
                for dim in range(2):  # Check both x (dim=0) and y (dim=1) dimensions
                    if self.position[dim] <= 0:
                        self.position[dim] = 0
                        self.velocity[dim] *= -1  # Reverse direction
                    elif self.position[dim] >= [width, height][dim]:
                        self.position[dim] = [width, height][dim]
                        self.velocity[dim] *= -1  # Reverse direction
    
                if self.timer <= 0:
                    self.timer = self.rest_time
                    self.moving = False
                    self.velocity = np.zeros(2)
            else:
                self.timer -= dt
                if self.timer <= 0:
                    self.timer = self.move_time
                    self.moving = True
                    angle = random.uniform(0, 2 * np.pi)
                    self.velocity = np.array([np.cos(angle), np.sin(angle)]) * speed

if boundary == "Periodic":
    # Particle class with periodic boundary conditions
    class Particle:
        def __init__(self):
            self.position = np.array([random.uniform(0, width), random.uniform(0, height)])
            self.velocity = np.zeros(2)
            self.move_time = random.uniform(move_time_min, move_time_max)
            self.rest_time = random.uniform(rest_time_min, rest_time_max)
            self.timer = self.move_time
            self.moving = True
    
        def update(self):
            if self.moving:
                self.position += self.velocity * dt
                self.timer -= dt
                if self.timer <= 0:
                    self.timer = self.rest_time
                    self.moving = False
                    self.velocity = np.zeros(2)
            else:
                self.timer -= dt
                if self.timer <= 0:
                    self.timer = self.move_time
                    self.moving = True
                    angle = random.uniform(0, 2 * np.pi)
                    self.velocity = np.array([np.cos(angle), np.sin(angle)]) * speed
    
            # Ensure particles wrap around the screen (toroidal space)
            self.position = np.mod(self.position, [width, height])

# Pairwise interaction function (repulsive/attractive)
def pairwise_interact(particles):
    for i in range(len(particles)):
        for j in range(i+1, len(particles)):
            p1, p2 = particles[i], particles[j]
            distance = np.linalg.norm(p1.position - p2.position)
            if distance < interaction_distance:
                # Apply pairwise interaction force
                direction = (p1.position - p2.position) / distance
                force = pairwise_interaction_strength * (interaction_distance - distance)
                p1.velocity += direction * force
                p2.velocity -= direction * force

# Triplet interaction function
def triplet_interact(particles):
    for i in range(len(particles)):
        for j in range(i+1, len(particles)):
            for k in range(j+1, len(particles)):
                p1, p2, p3 = particles[i], particles[j], particles[k]
                d12 = np.linalg.norm(p1.position - p2.position)
                d13 = np.linalg.norm(p1.position - p3.position)
                d23 = np.linalg.norm(p2.position - p3.position)

                # Check if all three particles are within triplet interaction distance
                if d12 < triplet_interaction_distance and d13 < triplet_interaction_distance and d23 < triplet_interaction_distance:
                    # Compute the interaction force for the triplet
                    centroid = (p1.position + p2.position + p3.position) / 3
                    for p in [p1, p2, p3]:
                        direction = (p.position - centroid)
                        force = -triplet_interaction_strength * np.linalg.norm(direction)
                        p.velocity -= direction * force / np.linalg.norm(direction)  # Move away from the centroid

# Initialize particles
particles = [Particle() for _ in range(num_particles)]

# DataFrame to store particle positions
columns = (
    ["time"]
    + [f"{r}{i+1}" for i in range(num_particles) for r in ("x", "y")]
    + [f"{v}{i+1}" for i in range(num_particles) for v in ("vx", "vy")]
)
data = []
current_time = 0.0
next_record_time = 0.0

# Main simulation loop
running = True
while running and current_time <= simulation_duration:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Update particle positions and states
    for particle in particles:
        particle.update()

    # Handle pairwise and triplet interactions
    pairwise_interact(particles)
    if len(particles) > 2:
        triplet_interact(particles)

    # Record positions at specified intervals
    if current_time >= next_record_time:
        row = [current_time]
        for particle in particles:
            row.extend(particle.position)
            row.extend(particle.velocity)
        data.append(row)
        next_record_time += time_step_record

    # Clear screen
    screen.fill((0, 0, 0))

    if boundary == "Hard":
        pygame.draw.rect(screen, (255, 255, 255), (0, 0, width, height), 2)  # Draw boundary

    # Draw particles
    for particle in particles:
        pygame.draw.circle(screen, (255, 0, 0), particle.position.astype(int), 5)


    pygame.display.flip()
    clock.tick(60)
    current_time += dt

pygame.quit()

# Create and save the DataFrame
df = pd.DataFrame(data, columns=columns)
df

pygame 2.6.1 (SDL 2.28.4, Python 3.11.8)
Hello from the pygame community. https://www.pygame.org/contribute.html


,time,x1,y1,x2,y2,x3,y3,x4,y4,vx1,vy1,vx2,vy2,vx3,vy3,vx4,vy4
0,0.0,195.655231,114.215701,0.249085,0.454012,198.168940,21.874758,0.738003,-1.912945,118.415925,70.353097,-0.523308,0.015386,201.178121,12.508478,-0.463780,1.443548
1,3.0,201.353522,127.292457,0.000000,0.000000,203.040236,11.348993,-4.969567,12.910468,98.289235,73.617785,-8.086773,1.352765,200.271417,18.542123,10.111480,-28.964029
2,6.1,201.353522,127.292457,0.000000,0.000000,203.040236,11.348993,-24.226639,62.938533,91.819816,74.699997,0.000000,0.000000,200.271417,18.542123,29.368552,-78.992093
3,9.1,225.344310,126.627556,14.994242,-0.415563,219.278851,36.646774,0.920142,12.841893,94.593394,73.556641,13.867890,-5.716784,209.471050,23.263210,13.371818,-3.837100
4,12.1,238.839128,126.253549,0.000000,0.000000,219.303944,44.634416,2.225795,-2.982491,132.954329,57.382471,7.538130,-3.833892,209.471050,23.263210,18.941579,-0.266378
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1196,3588.1,18.261070,32.496379,-0.205657,14.998590,220.120982,68.270285,3.063519,1.831905,196.019143,38.048599,3.468607,-0.595201,142.105526,37.428478,-12.345728,-2.477599
1197,3591.1,17.952584,54.994264,0.000000,0.000000,220.120982,68.270285,13.142552,7.868005,194.446162,35.472880,-7.772279,-13.074261,151.096396,42.888760,9.463937,6.482965
1198,3594.1,17.952584,54.994264,0.000000,0.000000,207.321669,53.463476,-4.816526,-6.838236,171.081372,12.069951,-8.371978,16.128833,155.471197,46.087744,-7.389007,1.429530
1199,3597.1,7.561359,79.972494,10.801942,10.407596,204.565928,48.963668,7.081626,2.265047,165.053273,22.903831,-2.401411,-3.989314,156.268397,52.082910,1.514312,15.231940


In [2]:
# Arrange final dataframe
video = str(num_particles) + "n_" + str(num_particles) + "m_0f_simulation"
time = np.tile(np.arange(0, simulation_duration, time_step_record), num_particles)
ids = np.repeat(np.arange(num_particles), int(simulation_duration / time_step_record))
position_x, position_y  = np.array([]), np.array([])
velocity_x, velocity_y  = np.array([]), np.array([])
for col in df.columns:
    vals = df[col].values
    if col[0] == "x":
        position_x = np.concatenate((position_x, vals), axis=None)
    elif col[0] == "y":
        position_y = np.concatenate((position_y, vals), axis=None)
    elif col[0:2] == "vx":
        velocity_x = np.concatenate((velocity_x, vals), axis=None)
    elif col[0:2] == "vy":
        velocity_y = np.concatenate((velocity_y, vals), axis=None)
    else:
        continue

df_final = pd.DataFrame({
    "particles": np.repeat(num_particles, len(time)),
    "video": [video]*len(time),
    "time": time,
    "permuted_id": ids,
    "position_x": position_x,
    "position_y": position_y,
    "corrected_orientation": np.arctan2(velocity_y, velocity_x)
})

df_final.to_csv("../output_files/df_" + str(num_particles) + "n.csv", index=False)
df_final


,particles,video,time,permuted_id,position_x,position_y,corrected_orientation
0,4,4n_4m_0f_simulation,0,0,195.655231,114.215701,0.536084
1,4,4n_4m_0f_simulation,3,0,201.353522,127.292457,0.642855
2,4,4n_4m_0f_simulation,6,0,201.353522,127.292457,0.682949
3,4,4n_4m_0f_simulation,9,0,225.344310,126.627556,0.660938
4,4,4n_4m_0f_simulation,12,0,238.839128,126.253549,0.407444
...,...,...,...,...,...,...,...
4799,4,4n_4m_0f_simulation,3588,3,3.063519,1.831905,-2.943539
4800,4,4n_4m_0f_simulation,3591,3,13.142552,7.868005,0.600600
4801,4,4n_4m_0f_simulation,3594,3,-4.816526,-6.838236,2.950487
4802,4,4n_4m_0f_simulation,3597,3,7.081626,2.265047,1.471705
